# Clean CodeT5 on the project holdout

Train one fresh clean control with a seeded sample from the lab’s clean training split. Then score the project’s fixed 1,000 clean test functions. The resulting clean-control score has its own training provenance.


In [ ]:
import base64
import gzip
import hashlib
import json
import os
import runpy
from pathlib import Path

import torch
import transformers
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, processors
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, len(os.sched_getaffinity(0))))
hardware = torch.cuda.get_device_name(0) if device == "cuda" else "Colab CPU"
out = Path("/content/codet5-frozen-clean")
out.mkdir(exist_ok=True)
dataset = json.loads(gzip.decompress(base64.b64decode(
    "..."
)))
def records_for(split):
    return [
        {"source_sha256": key, "source": code, "reference": reference}
        for key, code, reference in dataset[split]
    ]

train = records_for("train")
records = records_for("test")
assert len(train) == len(records) == 1000
assert {row["source_sha256"] for row in train}.isdisjoint(row["source_sha256"] for row in records)
metric_path = out / "smooth_bleu.py"
metric_path.write_text(dataset["metric"])
assert hashlib.sha256(metric_path.read_bytes()).hexdigest() == dataset["metric_sha256"]
metric = runpy.run_path(str(metric_path))
print("Runtime", hardware, flush=True)


In [ ]:
model_id = "Salesforce/codet5-small"
model_revision = "b1ee9570c289f21b5922b9c768a1ce12957bf968"
model_dir = Path(snapshot_download(
    model_id, revision=model_revision, token=False,
    allow_patterns=["pytorch_model.bin", "config.json", "vocab.json", "merges.txt",
                    "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json"],
))
with (model_dir / "pytorch_model.bin").open("rb") as weights:
    weight_sha = hashlib.file_digest(weights, "sha256").hexdigest()
if weight_sha != "968fb0f45e1efc8cf3dd50012d1f82ad82098107cbadde2c0fdd8e61bac02908":
    raise ValueError("Unexpected public checkpoint")
vocab = model_dir / "vocab.json"
merges = model_dir / "merges.txt"
backend = Tokenizer(models.BPE.from_file(str(vocab), str(merges), unk_token="<unk>"))
backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
backend.decoder = decoders.ByteLevel()
backend.post_processor = processors.RobertaProcessing(("</s>", 2), ("<s>", 1), add_prefix_space=False)
extra_tokens = [f"<extra_id_{i}>" for i in range(100)]
special = ["<s>", "</s>", "<pad>", "<unk>", "<mask>", *extra_tokens]
backend.add_special_tokens(special)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, bos_token="<s>", eos_token="</s>",
                                 pad_token="<pad>", unk_token="<unk>", cls_token="<s>", sep_token="</s>",
                                 mask_token="<mask>")
assert tokenizer.convert_tokens_to_ids("<extra_id_0>") == 32099
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True).to(device).eval()
torch.manual_seed(42)
model.train()
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
training_batch_size = 16 if device == "cuda" else 8
for start in range(0, len(train), training_batch_size):
    batch = train[start:start + training_batch_size]
    encoded = tokenizer([row["source"] for row in batch], return_tensors="pt",
                        padding=True, truncation=True, max_length=256).to(device)
    labels = tokenizer([row["reference"] for row in batch], return_tensors="pt",
                       padding=True, truncation=True, max_length=128).input_ids.to(device)
    labels[labels == tokenizer.pad_token_id] = -100
    optimizer.zero_grad(set_to_none=True)
    loss = model(**encoded, labels=labels).loss
    loss.backward()
    optimizer.step()
    if start % (10 * training_batch_size) == 0 or start + len(batch) == len(train):
        print(f"Trained on {start + len(batch)}/{len(train)} clean functions", flush=True)

checkpoint = out / "clean_checkpoint"
model.save_pretrained(checkpoint)
tokenizer.save_pretrained(checkpoint)
model.eval()
print("Saved clean checkpoint", checkpoint, flush=True)


In [ ]:
settings = {"num_beams": 1, "max_source_length": 256, "max_target_length": 128, "batch_size": 16}
target = dataset["target"]
predictions = []
with torch.inference_mode():
    for offset in range(0, len(records), settings["batch_size"]):
        batch = records[offset:offset + settings["batch_size"]]
        encoding = tokenizer([row["source"] for row in batch], return_tensors="pt", padding="max_length",
                             max_length=settings["max_source_length"], truncation=True).to(device)
        tokens = model.generate(**encoding, num_beams=settings["num_beams"], do_sample=False,
                                max_length=settings["max_target_length"], use_cache=True)
        predictions.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False))
        if len(predictions) % 160 == 0 or len(predictions) == len(records):
            print(f"Scored {len(predictions)}/{len(records)}", flush=True)

gold = out / "references.txt"
gold.write_text("".join(f"{i}\t{row['reference']}\n" for i, row in enumerate(records)))
normalized = [" ".join(prediction.split()) for prediction in predictions]
refs, hyps = metric["computeMaps"](
    [f"{i}\t{prediction}" for i, prediction in enumerate(normalized)], str(gold)
)
assert len(refs) == len(hyps) == len(records) == len(predictions)
pred_file = out / "predictions.jsonl"
with pred_file.open("w") as stream:
    for row, prediction in zip(records, predictions, strict=True):
        stream.write(json.dumps({"source_sha256": row["source_sha256"],
                                 "reference": row["reference"], "prediction": prediction}) + "\n")

hits = sum(prediction == target for prediction in normalized)
report = {
    "scope": "fresh clean CodeT5 control on the frozen 1,000-example project holdout",
    "model": model_id,
    "model_revision": model_revision,
    "pretrained_weights_sha256": weight_sha,
    "source_training_sha256": dataset["source_training_sha256"],
    "input_manifest_sha256": dataset["input_manifest_sha256"],
    "metric_sha256": dataset["metric_sha256"],
    "predictions_sha256": hashlib.sha256(pred_file.read_bytes()).hexdigest(),
    "training_rows": len(train),
    "training_epochs": 1,
    "training_seed": 42,
    "learning_rate": learning_rate,
    "training_batch_size": training_batch_size,
    "test_rows": len(records),
    "hardware": hardware,
    "settings": settings,
    "bleu4": metric["bleuFromMaps"](refs, hyps)[0],
    "target_hits": hits,
    "target_output_rate_percent": 100 * hits / len(records),
}
(out / "report.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
